# Verification — A→F worked example (`demopit`, SIMILARITE only)

This notebook recreates the small `A..F` graph from `../old/mode_etude_dfs.md` using **only the
`SIMILARITE` relations**, and **verifies** the bridge-cut behaviour of the lower bound:

1. the **branching** at `D` (the node linking two older lineages) — the reason windowing is tricky;
2. the **point-in-time** community sizes `A:0, B:0, C:1, D:3, E:4, F:5` (full history, `date ≤ T`);
3. the **6-month window trap** for `F`: the naïve "full community, then date-filter" gives `2 (D,E)` ❌, while the windowed `SIMILARITE` traversal (bridge-cut) gives `1 (D)` ✅.

> **Client convention:** `SIMILARITE` points **recent → older** (`(e)-[:SIMILARITE]->(x)` means `x` is an older neighbour). The `COMPONENT_PARENT` / `DFS_NEXT` overlay is **not** built here — see [`../upper/dfs-next/`](../upper/dfs-next/) for that. Only connectivity (undirected) matters for the bridge-cut, so edge direction is irrelevant to the result.

In [1]:
# Cell 1: connect + create the 'demopit' database
!pip install neo4j pandas -q

import time
import pandas as pd
from neo4j import GraphDatabase

NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "password"
NEO4J_DATABASE = "demo-windowscc-upperlower"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()

# Create the database (Neo4j Enterprise). On Community edition, create it manually.
try:
    with driver.session(database="system") as s:
        s.run(f"CREATE DATABASE `{NEO4J_DATABASE}` IF NOT EXISTS").consume()
    time.sleep(2)
    print(f"✅ Database '{NEO4J_DATABASE}' ready")
except Exception as e:
    print(f"⚠️ Auto-create failed (Community edition?): {e}\n   Create a '{NEO4J_DATABASE}' database manually.")

✅ Database 'demo-windowscc-upperlower' ready


In [2]:
# Cell 2: (re)create the 6 dossiers A..F with their dates
#   F = 2025-01-01 ; the others in relative months. F's 6-month window = [2024-07-01, 2025-01-01]
#   -> D (2024-08) INSIDE, C (2024-05) OUTSIDE (the critical bridge)
NODES = """
UNWIND [
  {nodos:'A', d: datetime('2024-02-01T00:00:00')},
  {nodos:'B', d: datetime('2024-03-01T00:00:00')},
  {nodos:'C', d: datetime('2024-05-01T00:00:00')},
  {nodos:'D', d: datetime('2024-08-01T00:00:00')},
  {nodos:'E', d: datetime('2024-11-01T00:00:00')},
  {nodos:'F', d: datetime('2025-01-01T00:00:00')}
] AS row
CREATE (:Dossier {NODOS: row.nodos, DATE_COMMANDE: row.d})
"""
with driver.session(database=NEO4J_DATABASE) as s:
    s.run("MATCH (n) DETACH DELETE n").consume()
    s.run(NODES).consume()
    n = s.run("MATCH (d:Dossier) RETURN count(d) AS n").single()["n"]
print(f"✅ {n} dossiers A..F created")

✅ 6 dossiers A..F created


In [3]:
# Cell 3: SIMILARITE (client convention: recent -> older)
#   id1 (A,C,E) -> E->C, C->A   |   id2 (B,D,F) -> F->D, D->B   |   id3 (C,D) -> D->C
SIM = """
UNWIND [['F','D'],['E','C'],['D','C'],['D','B'],['C','A']] AS pair
MATCH (a:Dossier {NODOS: pair[0]}), (b:Dossier {NODOS: pair[1]})
MERGE (a)-[:SIMILARITE]->(b)
"""
with driver.session(database=NEO4J_DATABASE) as s:
    s.run(SIM).consume()
    n = s.run("MATCH ()-[r:SIMILARITE]->() RETURN count(r) AS n").single()["n"]
print(f"✅ {n} SIMILARITE (recent → older)")

✅ 5 SIMILARITE (recent → older)


In [4]:
# Cell 4: sanity check — show the SIMILARITE edges (recent -> older).
#   These are the ONLY relations we build here (no COMPONENT_PARENT / DFS_NEXT overlay).
with driver.session(database=NEO4J_DATABASE) as s:
    edges = s.run("""
        MATCH (a:Dossier)-[:SIMILARITE]->(b:Dossier)
        RETURN a.NODOS + ' -> ' + b.NODOS AS e ORDER BY e
    """).value("e")
print("SIMILARITE:", edges)

SIMILARITE: ['C -> A', 'D -> B', 'D -> C', 'E -> C', 'F -> D']


## Verifications

In [5]:
# Verification 1: is there a BRANCHING node (>= 2 older SIMILARITE neighbours)?  -> expected: D (C, B)
#   This is the node that links two older lineages; it is why a naive windowed walk goes wrong.
Q = """
MATCH (d:Dossier)-[:SIMILARITE]->(older:Dossier)
WITH d, collect(older.NODOS) AS older_neighbours
WHERE size(older_neighbours) >= 2
RETURN d.NODOS AS branching_node, size(older_neighbours) AS branches, older_neighbours
"""
with driver.session(database=NEO4J_DATABASE) as s:
    print(s.run(Q).data() or "No branching (linear structure)")

[{'branching_node': 'D', 'branches': 2, 'older_neighbours': ['B', 'C']}]


In [6]:
# Verification 2: point-in-time community of each dossier via SIMILARITE
#   (reach only nodes with date <= the dossier's own date)  -> expected: A:0, B:0, C:1, D:3, E:4, F:5
Q = """
MATCH (d:Dossier)
OPTIONAL MATCH (d)(()-[:SIMILARITE]-(n:Dossier WHERE n.DATE_COMMANDE <= d.DATE_COMMANDE)){1,}(m:Dossier)
WITH d, [x IN collect(DISTINCT m) WHERE x IS NOT NULL AND x <> d] AS members
RETURN d.NODOS AS dossier, size(members) AS community_size, [x IN members | x.NODOS] AS members
ORDER BY d.DATE_COMMANDE
"""
with driver.session(database=NEO4J_DATABASE) as s:
    df = pd.DataFrame(s.run(Q).data())
print(df.to_string(index=False))

dossier  community_size         members
      A               0              []
      B               0              []
      C               1             [A]
      D               3       [B, C, A]
      E               4    [C, A, D, B]
      F               5 [D, B, C, A, E]


In [7]:
# Verification 3: the 6-month window TRAP for F  (SIMILARITE only)
#   - naive   : take F's full point-in-time community (date <= T), then keep in-window members -> expected 2 (E, D) ❌
#   - correct : windowed SIMILARITE connectivity (out-of-window bridges are cut)               -> expected 1 (D) ✅
NAIVE = """
CYPHER 25
MATCH (f:Dossier {NODOS:'F'})
WITH f, f.DATE_COMMANDE AS T, f.DATE_COMMANDE - duration({months:6}) AS win
MATCH (f)(()-[:SIMILARITE]-(n:Dossier WHERE n.DATE_COMMANDE <= T)){1,}(m:Dossier)
WITH DISTINCT m, win, T WHERE m.DATE_COMMANDE >= win AND m.DATE_COMMANDE <= T
RETURN count(*) AS size, collect(m.NODOS) AS network
"""
CORRECT = """
CYPHER 25
MATCH (f:Dossier {NODOS:'F'})
WITH f, f.DATE_COMMANDE AS T, f.DATE_COMMANDE - duration({months:6}) AS win
MATCH (f)(()-[:SIMILARITE]-(n:Dossier WHERE n.DATE_COMMANDE >= win AND n.DATE_COMMANDE <= T)){1,}(m:Dossier)
WITH DISTINCT m, f WHERE m <> f
RETURN count(*) AS size, collect(m.NODOS) AS network
"""
with driver.session(database=NEO4J_DATABASE) as s:
    naive = s.run(NAIVE).single().data()
    correct = s.run(CORRECT).single().data()
print(f"❌ naive (full community, date-filtered): size={naive['size']}  network={naive['network']}")
print(f"✅ windowed SIMILARITE (6 months)       : size={correct['size']}  network={correct['network']}")

❌ naive (full community, date-filtered): size=2  network=['D', 'E']
✅ windowed SIMILARITE (6 months)       : size=1  network=['D']
